# 01 · Exploración de los datos — Desempeño de las ventas

**Reto 1:** analizar el desempeño de las ventas y del personal de ventas (generación, tendencia, contribución y perfil para contratación).

Este notebook corresponde a la fase de **comprensión de los datos** de CRISP-DM. Aquí se revisa la calidad de las siete tablas, se toman las decisiones de limpieza y se buscan los patrones que guían el modelado. Las decisiones que salen de aquí se implementan en los pipelines de Kedro (`data_processing`, `feature_engineering` y `sales_analysis`), que son los que producen los resultados finales.

Los datos se leen desde el **catálogo de Kedro** (`conf/base/catalog.yml`), con las mismas rutas y opciones de lectura que los pipelines.

> Para ejecutarlo: copiar los CSV del curso en `data/01_raw/` y abrir el notebook con el kernel del entorno del proyecto.

**Contenido**
1. Carga de las tablas y validación de llaves
2. Faltantes, duplicados y columnas a recalcular
3. Distribuciones y outliers
4. Patrones de venta, vendedores y señal para el modelo
5. Conclusiones y decisiones de limpieza

## 1. Carga de las tablas y validación de llaves

Diagrama entidad-relación del dataset:

![ERD](../docs/erd.png)

In [1]:
import os
from pathlib import Path

import pandas as pd
from kedro.config import OmegaConfigLoader
from kedro.io import DataCatalog

# las rutas del catálogo son relativas a la raíz del proyecto
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)

# catálogo del proyecto (mismas rutas y opciones de lectura que los pipelines)
config = OmegaConfigLoader(
    str(PROJECT / "conf"), base_env="base", default_run_env="local"
)
catalog = DataCatalog.from_config(config["catalog"])

pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
names = [
    "sales",
    "products",
    "categories",
    "employees",
    "customers",
    "cities",
    "countries",
]
raw = {name: catalog.load(f"raw_{name}") for name in names}

pd.DataFrame(
    {
        "filas": {n: len(df) for n, df in raw.items()},
        "columnas": {n: df.shape[1] for n, df in raw.items()},
        "memoria (MB)": {
            n: df.memory_usage(deep=True).sum() / 1e6 for n, df in raw.items()
        },
    }
)

,filas,columnas,memoria (MB)
sales,6758125,9,776.48
products,452,9,0.06
categories,11,2,0.00
employees,23,8,0.00
customers,98759,6,7.92
cities,96,4,0.00
countries,206,3,0.01


`sales` es la tabla de hechos: cada fila es una línea de venta, con el vendedor (`SalesPersonID`), el cliente, el producto, la cantidad, el descuento y la fecha. Las demás tablas son dimensiones. Para el Reto 1 las más importantes son **`employees`** (perfil del vendedor) y **`products`** (precio para calcular el ingreso).

In [3]:
sales = raw["sales"]
sales.head()

,SalesID,SalesPersonID,CustomerID,ProductID,Quantity,Discount,TotalPrice,SalesDate,TransactionNumber
0,1,6,27039,381,7,0.00,0.00,2018-02-05 07:38:25.430,FQL4S94E4ME1EZFTG42G
1,2,16,25011,61,7,0.00,0.00,2018-02-02 16:03:31.150,12UGLX40DJ1A5DTFBHB8
2,3,13,94024,23,24,0.00,0.00,2018-05-03 19:31:56.880,5DT8RCPL87KI5EORO7B0
3,4,8,73966,176,19,0.20,0.00,2018-04-07 14:43:55.420,R3DR9MLD5NR76VO17ULE
4,5,10,32653,310,9,0.00,0.00,2018-02-12 15:37:03.940,4BGS0Z5OMAZ8NDAFHHP3


In [4]:
sales.dtypes.to_frame("tipo").T

,SalesID,SalesPersonID,CustomerID,ProductID,Quantity,Discount,TotalPrice,SalesDate,TransactionNumber
tipo,int64,int64,int64,int64,int64,float64,float64,str,str


`SalesDate` viene como texto, así que hay que convertirla a fecha. Ahora se valida que cada llave foránea tenga correspondencia en su tabla de origen: una venta con un producto, cliente o vendedor inexistente no podría analizarse.

In [5]:
def orphans(child, child_key, parent, parent_key):
    # filas de child cuya llave no existe en parent
    return int((~child[child_key].isin(parent[parent_key])).sum())


checks = [
    ("sales.ProductID → products", sales, "ProductID", raw["products"], "ProductID"),
    (
        "sales.CustomerID → customers",
        sales,
        "CustomerID",
        raw["customers"],
        "CustomerID",
    ),
    (
        "sales.SalesPersonID → employees",
        sales,
        "SalesPersonID",
        raw["employees"],
        "EmployeeID",
    ),
    (
        "products.CategoryID → categories",
        raw["products"],
        "CategoryID",
        raw["categories"],
        "CategoryID",
    ),
    ("customers.CityID → cities", raw["customers"], "CityID", raw["cities"], "CityID"),
    ("employees.CityID → cities", raw["employees"], "CityID", raw["cities"], "CityID"),
    (
        "cities.CountryID → countries",
        raw["cities"],
        "CountryID",
        raw["countries"],
        "CountryID",
    ),
]
pd.DataFrame(
    [(name, orphans(c, ck, p, pk)) for name, c, ck, p, pk in checks],
    columns=["relación", "llaves huérfanas"],
)

,relación,llaves huérfanas
0,sales.ProductID → products,0
1,sales.CustomerID → customers,0
2,sales.SalesPersonID → employees,0
3,products.CategoryID → categories,0
4,customers.CityID → cities,0
5,employees.CityID → cities,0
6,cities.CountryID → countries,0


In [6]:
print(
    "Vendedores en ventas:",
    sales["SalesPersonID"].nunique(),
    "de",
    len(raw["employees"]),
)
print("Productos vendidos:", sales["ProductID"].nunique(), "de", len(raw["products"]))
print(
    "Clientes con compras:", sales["CustomerID"].nunique(), "de", len(raw["customers"])
)
print("Países con ciudades:", raw["cities"]["CountryID"].unique())

Vendedores en ventas: 23 de 23
Productos vendidos: 452 de 452
Clientes con compras: 98759 de 98759
Países con ciudades: [32]


**Hallazgos de la sección 1**

- Las relaciones están **completas**: no hay llaves huérfanas en ninguna tabla. Los 23 vendedores, los 452 productos y los 98,759 clientes aparecen en las ventas.
- Todas las ciudades pertenecen a **un solo país** (`CountryID` 32), así que un análisis por país no aporta. La dimensión geográfica útil es la ciudad.
- `SalesDate` hay que convertirla a fecha.
- **Decisión:** la validación de llaves se repite en cada ejecución del pipeline (`data_processing.build_sales_primary`), que se detiene con error si aparece una llave huérfana en datos futuros.